# 05 — Exploratory Data Analysis / Visualization

Nuages de points, proportions de réussite et tendances par année.

Source : jeu historique IBM DS0321EN. Exécuter toutes les cellules dans Google Colab. Les sorties sont produites à l’exécution, pas préremplies.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from IPython.display import display
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
df = pd.read_csv(BASE + "dataset_part_2.csv")
assert {"Class", "FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "LandingPad"}.issubset(df.columns)
print('Dimensions:',df.shape); display(df.head()); display(df.isna().sum().to_frame('Missing'))

In [ ]:
def save_chart(name):
    plt.tight_layout(); plt.savefig(OUT/name,dpi=180,bbox_inches="tight"); plt.show()

sns.catplot(data=df,x="FlightNumber",y="LaunchSite",hue="Class",height=5,aspect=1.8)
plt.title("Flight Number vs Launch Site by Landing Class"); save_chart('flightnumber_launchsite.png')
sns.catplot(data=df,x="PayloadMass",y="LaunchSite",hue="Class",height=5,aspect=1.8)
plt.title("Payload Mass vs Launch Site"); save_chart('payload_launchsite.png')
sns.catplot(data=df,x="PayloadMass",y="Orbit",hue="Class",height=5,aspect=1.8)
plt.title("Payload Mass vs Orbit"); save_chart('payload_orbit.png')

In [ ]:
fig,ax=plt.subplots(figsize=(10,5)); sns.barplot(data=df,x="Orbit",y="Class",errorbar=None,ax=ax)
ax.set_ylabel('Proportion successful'); ax.set_title('Landing success rate by orbit'); ax.tick_params(axis='x',rotation=45)
save_chart('success_orbit.png')
fig,ax=plt.subplots(figsize=(10,5)); sns.scatterplot(data=df,x="FlightNumber",y="PayloadMass",hue="Class",ax=ax)
ax.set_title('Flight Number vs Payload Mass'); save_chart('flightnumber_payload.png')
df['Year']=pd.to_datetime(df['Date']).dt.year
by_year=df.groupby('Year')['Class'].agg(['size','mean']).rename(columns={'size':'Launches','mean':'SuccessRate'})
display(by_year); by_year.to_csv(OUT/'success_by_year.csv')
fig,ax=plt.subplots(figsize=(9,5)); by_year['mean' if 'mean' in by_year else 'SuccessRate'].plot(ax=ax,marker='o')
ax.set_ylim(0,1); ax.set_ylabel('Successful landing proportion'); ax.set_title('Landing success by year'); save_chart('success_by_year.png')

**Analyse :** comparer les taux et distributions observés. Une association descriptive ne prouve pas la causalité.

In [ ]:
import zipfile
with zipfile.ZipFile('05_EDA_Visualization_outputs.zip','w') as z:
    for p in OUT.iterdir(): z.write(p,arcname=p.name)
from google.colab import files
files.download('05_EDA_Visualization_outputs.zip')